
# PRUEBA TÉCNICA MDA - ARQUITECTURA COMERCIAL
## Máster Data Administrator
 **Fecha:**  26 de Enero del 2026  
**Autor:** Paola Andrea García Tangarife

configuración inicial de duckdb 🦆

In [18]:
# Importación de librerías
import duckdb
import pandas as pd

# Cargar archivos CSV
ventas = pd.read_csv('ventas_distribuidoras.csv')
vendedores = pd.read_csv('maestro_vendedores.csv')
metas = pd.read_csv('metas_mensuales.csv')

# REGISTRAR TABLAS DE REFERENCIA DEL ANEXO

# Tabla de distribuidoras del anexo
distribuidoras_ref = pd.DataFrame({
    'codigo_distribuidora': ['DIST-001', 'DIST-002', 'DIST-003', 'DIST-004', 'DIST-005',
                            'DIST-006', 'DIST-007', 'DIST-008', 'DIST-009', 'DIST-010'],
    'nombre_correcto': ['Distribuidora Norte S.A.', 'Comercializadora Central Ltda.',
                       'Distribuciones del Sur SAS', 'Logística Oriental SpA',
                       'Red Occidental de Distribución', 'Almacenes Unidos S.A.',
                       'Distribuidora Express Ltda.', 'Cadena Mayorista Nacional',
                       'Grupo Comercial Andino', 'Distribuciones Premium SAS'],
    'region_correcta': ['Norte', 'Centro', 'Sur', 'Oriente', 'Occidente',
                       'Centro', 'Norte', 'Sur', 'Oriente', 'Occidente']
})

# Tabla de categorías de vendedores del anexo
categorias_ref = pd.DataFrame({
    'categoria': ['Junior', 'Semi-Senior', 'Senior'],
    'antiguedad_min': [0, 3, 5],
    'antiguedad_max': [3, 5, 99],
    'comision_min': [1.5, 2.0, 3.0],
    'comision_max': [2.5, 3.5, 5.0]
})

# Conexión a DuckDB y registro de todas las tablas
con = duckdb.connect()

# Registrar DataFrames como tablas virtuales
con.register('ventas', ventas)
con.register('vendedores', vendedores)
con.register('metas', metas)
con.register('distribuidoras_ref', distribuidoras_ref)
con.register('categorias_ref', categorias_ref)



# 📊 PARTE 1: Exploración de Datos

**Objetivo:** Familiarizarse con los datos e identificar problemas de calidad.

In [19]:
# 1.1 ¿Cuántas transacciones hay por año?

query_1_1 = """
SELECT 
    año,
    COUNT(*) as total_transacciones,
    COUNT(*) FILTER (WHERE estado = 'Activo') as transacciones_activas,
    COUNT(*) FILTER (WHERE estado = 'Anulado') as transacciones_anuladas,
    ROUND(COUNT(*) FILTER (WHERE estado = 'Anulado') * 100.0 / COUNT(*), 2) as porcentaje_anuladas,
    ROUND(AVG(venta_neta), 2) as venta_promedio,
    SUM(venta_neta) as venta_total
FROM ventas
GROUP BY año
ORDER BY año
"""

result_1_1 = con.execute(query_1_1).fetchdf()
print("1.1 TRANSACCIONES POR AÑO:")
display(result_1_1)

1.1 TRANSACCIONES POR AÑO:


,año,total_transacciones,transacciones_activas,transacciones_anuladas,porcentaje_anuladas,venta_promedio,venta_total
0,2024,9539,8864,322,3.38,10340810.93,9.864100e+10
1,2025,9420,8517,518,5.50,11315825.44,1.065951e+11


In [20]:
# 1.2 ¿Cuántas transacciones tienen campos nulos en códigos clave?

query_1_2 = """
WITH null_analysis AS (
    SELECT 
        COUNT(*) as total_transacciones,
        COUNT(*) FILTER (WHERE codigo_distribuidora IS NULL) as nulos_distribuidora,
        COUNT(*) FILTER (WHERE codigo_agencia IS NULL) as nulos_agencia,
        COUNT(*) FILTER (WHERE codigo_vendedor IS NULL) as nulos_vendedor
    FROM ventas
)
SELECT 
    total_transacciones,
    nulos_distribuidora,
    nulos_agencia,
    nulos_vendedor,
    ROUND(nulos_distribuidora * 100.0 / total_transacciones, 2) as pct_nulos_distribuidora,
    ROUND(nulos_agencia * 100.0 / total_transacciones, 2) as pct_nulos_agencia,
    ROUND(nulos_vendedor * 100.0 / total_transacciones, 2) as pct_nulos_vendedor
FROM null_analysis
"""

result_1_2 = con.execute(query_1_2).fetchdf()
print("1.2 ANÁLISIS DE VALORES NULOS:")
display(result_1_2)

1.2 ANÁLISIS DE VALORES NULOS:


,total_transacciones,nulos_distribuidora,nulos_agencia,nulos_vendedor,pct_nulos_distribuidora,pct_nulos_agencia,pct_nulos_vendedor
0,18959,602,788,629,3.18,4.16,3.32


In [21]:
# 1.3 ¿Existen registros duplicados?

query_1_3 = """
SELECT 
    COUNT(*) as total_ids,
    COUNT(DISTINCT id_transaccion) as ids_unicos,
    COUNT(*) - COUNT(DISTINCT id_transaccion) as duplicados,
    CASE 
        WHEN COUNT(*) = COUNT(DISTINCT id_transaccion) THEN '✅ Sin duplicados'
        ELSE '⚠️ Con duplicados'
    END as estado
FROM ventas
"""

result_1_3 = con.execute(query_1_3).fetchdf()
print("1.3 ANÁLISIS DE DUPLICADOS:")
display(result_1_3)

1.3 ANÁLISIS DE DUPLICADOS:


,total_ids,ids_unicos,duplicados,estado
0,18959,18959,0,✅ Sin duplicados


In [22]:
# 1.4 ¿Cuál es el resumen de ventas por distribuidora y año?

query_1_4 = """
SELECT 
    codigo_distribuidora,
    nombre_distribuidora,
    año,
    COUNT(*) as total_transacciones,
    SUM(unidades) as unidades_totales,
    ROUND(SUM(venta_neta), 2) as venta_neta_total,
    ROUND(AVG(venta_neta), 2) as venta_neta_promedio
FROM ventas
WHERE estado = 'Activo' 
    AND codigo_distribuidora IS NOT NULL
GROUP BY codigo_distribuidora, nombre_distribuidora, año
ORDER BY año, venta_neta_total DESC
"""

result_1_4 = con.execute(query_1_4).fetchdf()
print("1.4 RESUMEN DE VENTAS POR DISTRIBUIDORA Y AÑO (Top 10):")
display(result_1_4.head(10))

1.4 RESUMEN DE VENTAS POR DISTRIBUIDORA Y AÑO (Top 10):


,codigo_distribuidora,nombre_distribuidora,año,total_transacciones,unidades_totales,venta_neta_total,venta_neta_promedio
0,DIST-003,Distribuciones del Sur SAS,2024,1325,855343.0,7.533975e+10,56860190.74
1,DIST-008,Cadena Mayorista Nacional,2024,1147,270167.0,2.240435e+09,1953300.05
2,DIST-010,Distribuciones Premium SAS,2024,965,228930.0,1.934517e+09,2004680.64
3,DIST-002,Comercializadora Central Ltda.,2024,915,215990.0,1.799883e+09,1967085.41
4,DIST-007,Distribuidora Express Ltda.,2024,824,193796.0,1.584440e+09,1922863.49
5,DIST-004,Logística Oriental SpA,2024,783,186253.0,1.574586e+09,2010965.18
6,DIST-001,Distribuidora Norte S.A.,2024,734,173339.0,1.452555e+09,1978957.93
7,DIST-005,Red Occidental de Distribución,2024,730,169534.0,1.349159e+09,1848162.42
8,DIST-006,Almacenes Unidos S.A.,2024,691,162255.0,1.348973e+09,1952204.08
9,DIST-009,Grupo Comercial Andino,2024,597,139532.0,1.131730e+09,1895694.96


In [23]:
# VALIDACIÓN CON TABLAS DE REFERENCIA DEL ANEXO
# ====================================================

# Validación 1: Distribuidoras vs referencia
validacion_dist = con.execute("""
WITH distribuidoras_ventas AS (
    SELECT DISTINCT 
        codigo_distribuidora,
        nombre_distribuidora,
        region
    FROM ventas
    WHERE codigo_distribuidora IS NOT NULL
)
SELECT 
    'Distribuidoras en ventas vs referencia' as validacion,
    COUNT(*) as total_distribuidoras,
    COUNT(*) FILTER (WHERE dr.nombre_correcto IS NOT NULL) as coinciden_con_referencia,
    COUNT(*) FILTER (WHERE dr.nombre_correcto IS NULL) as no_en_referencia,
    ROUND(COUNT(*) FILTER (WHERE dr.nombre_correcto IS NOT NULL) * 100.0 / COUNT(*), 2) as porcentaje_coincidencia
FROM distribuidoras_ventas dv
LEFT JOIN distribuidoras_ref dr ON dv.codigo_distribuidora = dr.codigo_distribuidora
""").fetchdf()

print("VALIDACIÓN DISTRIBUIDORAS vs REFERENCIA:")
display(validacion_dist)

# Validación 2: Comisiones vs rangos de referencia
validacion_comisiones = con.execute("""
WITH vendedores_info AS (
    SELECT 
        codigo_vendedor,
        nombre_vendedor,
        categoria,
        comision_pct,
        EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM DATE(fecha_ingreso)) as antiguedad_anios
    FROM vendedores
    WHERE activo = TRUE
)
SELECT 
    'Comisiones vs rangos de referencia' as validacion,
    COUNT(*) as total_vendedores,
    COUNT(*) FILTER (WHERE comision_pct BETWEEN cr.comision_min AND cr.comision_max 
                     AND antiguedad_anios BETWEEN cr.antiguedad_min AND cr.antiguedad_max) as dentro_de_rango,
    COUNT(*) FILTER (WHERE comision_pct NOT BETWEEN cr.comision_min AND cr.comision_max 
                     OR antiguedad_anios NOT BETWEEN cr.antiguedad_min AND cr.antiguedad_max) as fuera_de_rango,
    ROUND(COUNT(*) FILTER (WHERE comision_pct BETWEEN cr.comision_min AND cr.comision_max 
                     AND antiguedad_anios BETWEEN cr.antiguedad_min AND cr.antiguedad_max) * 100.0 / COUNT(*), 2) as porcentaje_ok
FROM vendedores_info vi
LEFT JOIN categorias_ref cr ON vi.categoria = cr.categoria
""").fetchdf()

print("\nVALIDACIÓN COMISIONES vs RANGOS DE REFERENCIA:")
display(validacion_comisiones)

VALIDACIÓN DISTRIBUIDORAS vs REFERENCIA:


,validacion,total_distribuidoras,coinciden_con_referencia,no_en_referencia,porcentaje_coincidencia
0,Distribuidoras en ventas vs referencia,10,10,0,100.0



VALIDACIÓN COMISIONES vs RANGOS DE REFERENCIA:


,validacion,total_vendedores,dentro_de_rango,fuera_de_rango,porcentaje_ok
0,Comisiones vs rangos de referencia,30,22,8,73.33


# 🧹 PARTE 2: Limpieza y Cruce de Datos

**Objetivo:** Identificar y resolver inconsistencias para permitir el cruce correcto entre tablas.

In [24]:
# 2.1 Identificar inconsistencias en codigo_distribuidora de la tabla de metas

query_2_1 = """
SELECT 
    codigo_distribuidora,
    COUNT(*) as cantidad_registros,
    CASE 
        WHEN codigo_distribuidora LIKE 'DIST-%' AND LENGTH(codigo_distribuidora) = 8 THEN '✅ Formato correcto'
        WHEN codigo_distribuidora LIKE 'DIST-%' THEN '⚠️ Formato incompleto'
        ELSE '❌ Formato incorrecto'
    END as estado_formato
FROM metas
GROUP BY codigo_distribuidora
ORDER BY estado_formato, codigo_distribuidora
"""

result_2_1 = con.execute(query_2_1).fetchdf()
print("2.1 INCONSISTENCIAS EN CÓDIGOS DE DISTRIBUIDORA:")
display(result_2_1)

2.1 INCONSISTENCIAS EN CÓDIGOS DE DISTRIBUIDORA:


,codigo_distribuidora,cantidad_registros,estado_formato
0,DIST-001,51,✅ Formato correcto
1,DIST-002,66,✅ Formato correcto
2,DIST-003,90,✅ Formato correcto
3,DIST-004,59,✅ Formato correcto
4,DIST-005,76,✅ Formato correcto
5,DIST-006,58,✅ Formato correcto
6,DIST-007,64,✅ Formato correcto
7,DIST-008,94,✅ Formato correcto
8,DIST-009,53,✅ Formato correcto
9,DIST-010,65,✅ Formato correcto


In [25]:
# 2.2 Consulta SQL que normaliza los códigos para poder hacer el cruce

print("2.2 NORMALIZACIÓN DE CÓDIGOS:")

create_view_query = """
CREATE OR REPLACE VIEW metas_normalizadas AS
SELECT 
    año,
    mes,
    periodo,
    -- NORMALIZACIÓN COMPLETA DE CÓDIGOS:
    CASE 
        -- Corrección de guión bajo a guión normal
        WHEN codigo_distribuidora = 'DIST_001' THEN 'DIST-001'
        WHEN codigo_distribuidora = 'DIST_002' THEN 'DIST-002'
        WHEN codigo_distribuidora = 'DIST_003' THEN 'DIST-003'
        WHEN codigo_distribuidora = 'DIST_004' THEN 'DIST-004'
        WHEN codigo_distribuidora = 'DIST_005' THEN 'DIST-005'
        WHEN codigo_distribuidora = 'DIST_006' THEN 'DIST-006'
        WHEN codigo_distribuidora = 'DIST_007' THEN 'DIST-007'
        WHEN codigo_distribuidora = 'DIST_008' THEN 'DIST-008'
        WHEN codigo_distribuidora = 'DIST_009' THEN 'DIST-009'
        WHEN codigo_distribuidora = 'DIST_010' THEN 'DIST-010'
        -- Corrección de formatos incompletos
        WHEN codigo_distribuidora = 'DIST-01' THEN 'DIST-001'
        WHEN codigo_distribuidora = 'DIST-02' THEN 'DIST-002'
        WHEN codigo_distribuidora = 'DIST-03' THEN 'DIST-003'
        WHEN codigo_distribuidora = 'DIST-04' THEN 'DIST-004'
        WHEN codigo_distribuidora = 'DIST-05' THEN 'DIST-005'
        WHEN codigo_distribuidora = 'DIST-06' THEN 'DIST-006'
        WHEN codigo_distribuidora = 'DIST-07' THEN 'DIST-007'
        WHEN codigo_distribuidora = 'DIST-08' THEN 'DIST-008'
        WHEN codigo_distribuidora = 'DIST-09' THEN 'DIST-009'
        WHEN codigo_distribuidora = 'DIST-10' THEN 'DIST-010'
        ELSE codigo_distribuidora
    END as codigo_distribuidora,
    codigo_agencia,
    meta_venta,
    meta_unidades
FROM metas
"""

con.execute(create_view_query)
print("✅ Vista 'metas_normalizadas' creada exitosamente")

2.2 NORMALIZACIÓN DE CÓDIGOS:
✅ Vista 'metas_normalizadas' creada exitosamente


In [26]:
# Verificar la normalización
verification_query = """
SELECT 
    'ANTES' as etapa,
    codigo_distribuidora as codigo,
    COUNT(*) as registros
FROM metas
WHERE codigo_distribuidora IN ('DIST_001', 'DIST_002', 'DIST-01', 'DIST-02')
GROUP BY codigo_distribuidora

UNION ALL

SELECT 
    'DESPUÉS' as etapa,
    codigo_distribuidora as codigo,
    COUNT(*) as registros
FROM metas_normalizadas
WHERE codigo_distribuidora IN ('DIST-001', 'DIST-002')
GROUP BY codigo_distribuidora

ORDER BY etapa, codigo
"""

verification = con.execute(verification_query).fetchdf()
print("VERIFICACIÓN DE NORMALIZACIÓN:")
display(verification)

VERIFICACIÓN DE NORMALIZACIÓN:


,etapa,codigo,registros
0,ANTES,DIST_001,21
1,ANTES,DIST_002,30
2,DESPUÉS,DIST-001,72
3,DESPUÉS,DIST-002,96


In [27]:
# 2.3 Realizar el cruce entre ventas y metas

query_cruce = """
WITH ventas_agrupadas AS (
    SELECT 
        año,
        mes,
        codigo_distribuidora,
        codigo_agencia,
        SUM(venta_neta) as venta_real_total
    FROM ventas
    WHERE estado = 'Activo'
        AND codigo_distribuidora IS NOT NULL
        AND codigo_agencia IS NOT NULL
    GROUP BY año, mes, codigo_distribuidora, codigo_agencia
),
metas_preparadas AS (
    SELECT 
        año,
        mes,
        codigo_distribuidora,
        codigo_agencia,
        meta_venta
    FROM metas_normalizadas
),
cruce_completo AS (
    SELECT 
        COALESCE(v.año, m.año) as año,
        COALESCE(v.mes, m.mes) as mes,
        COALESCE(v.codigo_distribuidora, m.codigo_distribuidora) as codigo_distribuidora,
        COALESCE(v.codigo_agencia, m.codigo_agencia) as codigo_agencia,
        CASE WHEN v.codigo_distribuidora IS NOT NULL THEN 1 ELSE 0 END as tiene_ventas,
        CASE WHEN m.codigo_distribuidora IS NOT NULL THEN 1 ELSE 0 END as tiene_metas
    FROM ventas_agrupadas v
    FULL OUTER JOIN metas_preparadas m 
        ON v.año = m.año 
        AND v.mes = m.mes 
        AND v.codigo_distribuidora = m.codigo_distribuidora
        AND v.codigo_agencia = m.codigo_agencia
)
SELECT 
    CASE 
        WHEN tiene_ventas = 1 AND tiene_metas = 1 THEN '✅ Cruce exitoso'
        WHEN tiene_ventas = 1 AND tiene_metas = 0 THEN '⚠️ Solo ventas (sin metas)'
        WHEN tiene_ventas = 0 AND tiene_metas = 1 THEN '⚠️ Solo metas (sin ventas)'
        ELSE '❌ Sin datos'
    END as estado_cruce,
    COUNT(*) as cantidad_registros,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as porcentaje
FROM cruce_completo
GROUP BY estado_cruce
ORDER BY cantidad_registros DESC
"""

cruce_result = con.execute(query_cruce).fetchdf()
print("2.3 RESULTADO DEL CRUCE VENTAS-METAS:")
display(cruce_result)

2.3 RESULTADO DEL CRUCE VENTAS-METAS:


,estado_cruce,cantidad_registros,porcentaje
0,✅ Cruce exitoso,912,100.0


In [28]:
# 2.4 CTE con datos de cumplimiento de metas

cumplimiento_query = """
CREATE OR REPLACE VIEW vista_cumplimiento AS
WITH ventas_agrupadas AS (
    SELECT 
        año,
        mes,
        codigo_distribuidora,
        codigo_agencia,
        SUM(venta_neta) as venta_real,
        SUM(unidades) as unidades_reales
    FROM ventas
    WHERE estado = 'Activo'
        AND codigo_distribuidora IS NOT NULL
        AND codigo_agencia IS NOT NULL
    GROUP BY año, mes, codigo_distribuidora, codigo_agencia
),
metas_preparadas AS (
    SELECT 
        año,
        mes,
        codigo_distribuidora,
        codigo_agencia,
        meta_venta,
        meta_unidades
    FROM metas_normalizadas
)
SELECT 
    COALESCE(v.año, m.año) as año,
    COALESCE(v.mes, m.mes) as mes,
    COALESCE(v.codigo_distribuidora, m.codigo_distribuidora) as codigo_distribuidora,
    COALESCE(v.codigo_agencia, m.codigo_agencia) as codigo_agencia,
    COALESCE(v.venta_real, 0) as venta_real,
    COALESCE(m.meta_venta, 0) as meta_venta,
    COALESCE(v.unidades_reales, 0) as unidades_reales,
    COALESCE(m.meta_unidades, 0) as meta_unidades,
    CASE 
        WHEN COALESCE(m.meta_venta, 0) = 0 THEN NULL
        ELSE ROUND((COALESCE(v.venta_real, 0) * 100.0 / m.meta_venta), 2)
    END as porcentaje_cumplimiento_venta,
    CASE 
        WHEN COALESCE(m.meta_unidades, 0) = 0 THEN NULL
        ELSE ROUND((COALESCE(v.unidades_reales, 0) * 100.0 / m.meta_unidades), 2)
    END as porcentaje_cumplimiento_unidades
FROM ventas_agrupadas v
FULL OUTER JOIN metas_preparadas m 
    ON v.año = m.año 
    AND v.mes = m.mes 
    AND v.codigo_distribuidora = m.codigo_distribuidora
    AND v.codigo_agencia = m.codigo_agencia
"""

con.execute(cumplimiento_query)
print("2.4 ✅ Vista 'vista_cumplimiento' creada exitosamente")

2.4 ✅ Vista 'vista_cumplimiento' creada exitosamente


# 📈 PARTE 3: Análisis de Negocio con SQL

**Objetivo:** Construir consultas SQL que respondan preguntas clave de la gerencia.

In [29]:
# QUERY 1: Top 10 Distribuidoras por Venta Neta 2024 (CON REFERENCIA DEL ANEXO)

query_1 = """
WITH ventas_2024 AS (
    SELECT 
        v.codigo_distribuidora,
        COALESCE(dr.nombre_correcto, v.nombre_distribuidora) as nombre_distribuidora,
        COALESCE(dr.region_correcta, v.region) as region,
        COUNT(*) as cantidad_transacciones,
        SUM(v.venta_neta) as venta_neta_total
    FROM ventas v
    LEFT JOIN distribuidoras_ref dr ON v.codigo_distribuidora = dr.codigo_distribuidora
    WHERE v.año = 2024 
        AND v.estado = 'Activo'
        AND v.codigo_distribuidora IS NOT NULL
    GROUP BY v.codigo_distribuidora, dr.nombre_correcto, v.nombre_distribuidora, dr.region_correcta, v.region
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY venta_neta_total DESC) as ranking,
    nombre_distribuidora,
    region,
    ROUND(venta_neta_total, 2) as venta_neta_total,
    cantidad_transacciones,
    ROUND(venta_neta_total / cantidad_transacciones, 2) as promedio_por_transaccion
FROM ventas_2024
ORDER BY venta_neta_total DESC
LIMIT 10
"""

result_1 = con.execute(query_1).fetchdf()
print("QUERY 1: TOP 10 DISTRIBUIDORAS POR VENTA NETA 2024 (CON REFERENCIA):")
display(result_1)

QUERY 1: TOP 10 DISTRIBUIDORAS POR VENTA NETA 2024 (CON REFERENCIA):


,ranking,nombre_distribuidora,region,venta_neta_total,cantidad_transacciones,promedio_por_transaccion
0,1,Distribuciones del Sur SAS,Sur,7.533975e+10,1325,56860190.74
1,2,Cadena Mayorista Nacional,Sur,2.240435e+09,1147,1953300.05
2,3,Distribuciones Premium SAS,Occidente,1.934517e+09,965,2004680.64
3,4,Comercializadora Central Ltda.,Centro,1.799883e+09,915,1967085.41
4,5,Distribuidora Express Ltda.,Norte,1.584440e+09,824,1922863.49
5,6,Logística Oriental SpA,Oriente,1.574586e+09,783,2010965.18
6,7,Distribuidora Norte S.A.,Norte,1.452555e+09,734,1978957.93
7,8,Red Occidental de Distribución,Occidente,1.349159e+09,730,1848162.42
8,9,Almacenes Unidos S.A.,Centro,1.348973e+09,691,1952204.08
9,10,Grupo Comercial Andino,Oriente,1.131730e+09,597,1895694.96


In [30]:
# QUERY 2: Cumplimiento de Metas por Agencia (CON REFERENCIA)

query_2 = """
SELECT 
    vc.año,
    vc.mes,
    vc.codigo_distribuidora,
    COALESCE(dr.nombre_correcto, d.nombre_distribuidora) as nombre_distribuidora,
    vc.codigo_agencia,
    vc.venta_real,
    vc.meta_venta,
    vc.porcentaje_cumplimiento_venta,
    CASE 
        WHEN vc.porcentaje_cumplimiento_venta IS NULL THEN 'Sin meta definida'
        WHEN vc.porcentaje_cumplimiento_venta > 100 THEN '✅ Sobrecumplimiento'
        WHEN vc.porcentaje_cumplimiento_venta >= 80 THEN '✅ Cumplimiento'
        ELSE '⚠️ Incumplimiento'
    END as clasificacion_cumplimiento
FROM vista_cumplimiento vc
LEFT JOIN (
    SELECT DISTINCT codigo_distribuidora, nombre_distribuidora
    FROM ventas
    WHERE codigo_distribuidora IS NOT NULL
) d ON vc.codigo_distribuidora = d.codigo_distribuidora
LEFT JOIN distribuidoras_ref dr ON vc.codigo_distribuidora = dr.codigo_distribuidora
WHERE vc.porcentaje_cumplimiento_venta IS NOT NULL
    AND vc.año = 2024
ORDER BY vc.año DESC, vc.mes DESC, vc.porcentaje_cumplimiento_venta DESC
"""

result_2 = con.execute(query_2).fetchdf()
print("QUERY 2: CUMPLIMIENTO DE METAS POR AGENCIA (2024 - CON REFERENCIA):")
print(f"Total registros: {len(result_2):,}")
display(result_2.head(10))

QUERY 2: CUMPLIMIENTO DE METAS POR AGENCIA (2024 - CON REFERENCIA):
Total registros: 456


,año,mes,codigo_distribuidora,nombre_distribuidora,codigo_agencia,venta_real,meta_venta,porcentaje_cumplimiento_venta,clasificacion_cumplimiento
0,2024,12,DIST-001,Distribuidora Norte S.A.,AGN-102,57934935.56,49184610.0,117.79,✅ Sobrecumplimiento
1,2024,12,DIST-006,Almacenes Unidos S.A.,AGU-602,52382065.91,44807479.0,116.90,✅ Sobrecumplimiento
2,2024,12,DIST-010,Distribuciones Premium SAS,AGP-1004,57934935.56,57814372.0,100.21,✅ Sobrecumplimiento
3,2024,12,DIST-010,Distribuciones Premium SAS,AGP-1003,52382065.91,54400724.0,96.29,✅ Cumplimiento
4,2024,12,DIST-007,Distribuidora Express Ltda.,AGE-701,52382065.91,62300504.0,84.08,✅ Cumplimiento
5,2024,12,DIST-008,Cadena Mayorista Nacional,AGM-802,39369930.31,50123354.0,78.55,⚠️ Incumplimiento
6,2024,12,DIST-003,Distribuciones del Sur SAS,AGS-301,39369930.31,52079289.0,75.60,⚠️ Incumplimiento
7,2024,12,DIST-010,Distribuciones Premium SAS,AGP-1001,52382065.91,71408344.0,73.36,⚠️ Incumplimiento
8,2024,12,DIST-001,Distribuidora Norte S.A.,AGN-101,57934935.56,88175955.0,65.70,⚠️ Incumplimiento
9,2024,12,DIST-009,Grupo Comercial Andino,AGA-902,52382065.91,82577704.0,63.43,⚠️ Incumplimiento


In [31]:
# QUERY 3: Análisis de Vendedores (Top 10) - CON VALIDACIÓN DE REFERENCIA

# Primero crear una vista para ventas por vendedor 2024
con.execute("""
CREATE OR REPLACE TEMP VIEW ventas_vendedor_2024 AS
SELECT 
    codigo_vendedor,
    SUM(venta_neta) as venta_neta_total,
    COUNT(*) as cantidad_transacciones
FROM ventas
WHERE estado = 'Activo'
    AND año = 2024
    AND codigo_vendedor IS NOT NULL
GROUP BY codigo_vendedor
""")

query_3 = """
WITH vendedores_info AS (
    SELECT 
        vv.codigo_vendedor,
        vv.venta_neta_total,
        vv.cantidad_transacciones,
        vend.nombre_vendedor,
        vend.categoria,
        vend.comision_pct,
        vend.fecha_ingreso,
        -- Calcular antigüedad correctamente
        EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM DATE(vend.fecha_ingreso)) as antiguedad_anios,
        vend.activo,
        -- Calcular comisión
        ROUND((vv.venta_neta_total * vend.comision_pct / 100), 2) as comision_ganada,
        -- Validar vs referencia del anexo
        CASE 
            WHEN vend.comision_pct BETWEEN cr.comision_min AND cr.comision_max 
                 AND (EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM DATE(vend.fecha_ingreso))) 
                     BETWEEN cr.antiguedad_min AND cr.antiguedad_max
            THEN '✅ DENTRO DE RANGO'
            WHEN vend.comision_pct < cr.comision_min THEN '⚠️ COMISIÓN BAJA'
            WHEN vend.comision_pct > cr.comision_max THEN '⚠️ COMISIÓN ALTA'
            ELSE '❌ FUERA DE RANGO'
        END as estado_validacion
    FROM ventas_vendedor_2024 vv
    INNER JOIN vendedores vend ON vv.codigo_vendedor = vend.codigo_vendedor
    LEFT JOIN categorias_ref cr ON vend.categoria = cr.categoria
    WHERE vend.activo = TRUE
)
SELECT 
    ROW_NUMBER() OVER (ORDER BY venta_neta_total DESC) as ranking,
    nombre_vendedor,
    categoria,
    antiguedad_anios,
    ROUND(venta_neta_total, 2) as venta_neta_total,
    cantidad_transacciones,
    ROUND(venta_neta_total / cantidad_transacciones, 2) as promedio_venta,
    comision_pct,
    ROUND(comision_ganada, 2) as comision_ganada,
    estado_validacion
FROM vendedores_info
ORDER BY venta_neta_total DESC
LIMIT 10
"""

result_3 = con.execute(query_3).fetchdf()
print("QUERY 3: TOP 10 VENDEDORES POR VENTA NETA 2024 (CON VALIDACIÓN DE REFERENCIA):")
display(result_3)

QUERY 3: TOP 10 VENDEDORES POR VENTA NETA 2024 (CON VALIDACIÓN DE REFERENCIA):


,ranking,nombre_vendedor,categoria,antiguedad_anios,venta_neta_total,cantidad_transacciones,promedio_venta,comision_pct,comision_ganada,estado_validacion
0,1,Ana Martínez,Senior,8,1.089134e+10,380,28661410.63,3.14,3.419880e+08,✅ DENTRO DE RANGO
1,2,Pedro Rodríguez,Junior,4,1.063178e+10,350,30376503.13,1.64,1.743611e+08,❌ FUERA DE RANGO
2,3,Paula Vega,Semi-Senior,5,6.640382e+09,598,11104318.02,2.88,1.912430e+08,✅ DENTRO DE RANGO
3,4,Fernando Cruz,Semi-Senior,6,6.251978e+09,442,14144746.16,2.50,1.562994e+08,❌ FUERA DE RANGO
4,5,Carolina Reyes,Junior,2,6.247668e+09,407,15350536.72,1.82,1.137076e+08,✅ DENTRO DE RANGO
5,6,Tomás Aguilar,Junior,2,5.258607e+09,382,13765987.36,2.36,1.241031e+08,✅ DENTRO DE RANGO
6,7,Sofía Torres,Junior,2,5.144569e+09,391,13157465.37,1.82,9.363116e+07,✅ DENTRO DE RANGO
7,8,Felipe Vargas,Semi-Senior,5,4.962674e+09,331,14992974.17,2.67,1.325034e+08,✅ DENTRO DE RANGO
8,9,Diego Hernández,Semi-Senior,6,4.855913e+09,250,19423650.58,2.27,1.102292e+08,❌ FUERA DE RANGO
9,10,Mariana Navarro,Junior,2,4.841692e+09,217,22311946.91,2.02,9.780219e+07,✅ DENTRO DE RANGO


# 💾 PARTE 4: Sábana Enriquecida

**Objetivo:** Construir una consulta final que integre toda la información en una sábana lista para tableros.

In [32]:
# Crear sábana enriquecida CON REFERENCIAS DEL ANEXO
sabana_query = """
WITH ventas_activas AS (
    SELECT *
    FROM ventas
    WHERE estado = 'Activo'
        AND codigo_vendedor IS NOT NULL
        AND codigo_distribuidora IS NOT NULL
),
metas_preparadas AS (
    SELECT 
        año,
        mes,
        codigo_distribuidora,
        codigo_agencia,
        meta_venta
    FROM metas_normalizadas
),
ventas_con_metas AS (
    SELECT 
        v.*,
        COALESCE(m.meta_venta, 0) as meta_mensual_agencia
    FROM ventas_activas v
    LEFT JOIN metas_preparadas m 
        ON v.año = m.año 
        AND v.mes = m.mes 
        AND v.codigo_distribuidora = m.codigo_distribuidora
        AND v.codigo_agencia = m.codigo_agencia
)
SELECT 
    -- Información de transacción
    vm.id_transaccion,
    vm.fecha,
    vm.año,
    vm.mes,
    
    -- Información de distribuidora (USANDO REFERENCIA DEL ANEXO)
    vm.codigo_distribuidora,
    COALESCE(dr.nombre_correcto, vm.nombre_distribuidora) as nombre_distribuidora,
    COALESCE(dr.region_correcta, vm.region) as region,
    vm.codigo_agencia,
    
    -- Información del vendedor (CON VALIDACIÓN DE REFERENCIA)
    vm.codigo_vendedor,
    vend.nombre_vendedor,
    vend.categoria as categoria_vendedor,
    vend.fecha_ingreso,
    EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM DATE(vend.fecha_ingreso)) as antiguedad_anios,
    vend.comision_pct,
    vend.activo as vendedor_activo,
    
    -- Validación vs referencia del anexo
    CASE 
        WHEN vend.comision_pct BETWEEN cr.comision_min AND cr.comision_max 
             AND (EXTRACT(YEAR FROM CURRENT_DATE) - EXTRACT(YEAR FROM DATE(vend.fecha_ingreso))) 
                 BETWEEN cr.antiguedad_min AND cr.antiguedad_max
        THEN '✅ DENTRO DE RANGO'
        WHEN vend.comision_pct < cr.comision_min THEN '⚠️ COMISIÓN BAJA'
        WHEN vend.comision_pct > cr.comision_max THEN '⚠️ COMISIÓN ALTA'
        ELSE '❌ FUERA DE RANGO'
    END as estado_validacion_comision,
    
    -- Información del producto
    vm.codigo_producto,
    vm.nombre_producto,
    vm.categoria as categoria_producto,
    vm.canal_venta,
    vm.tipo_cliente,
    
    -- Métricas financieras
    ROUND(vm.venta_neta, 2) as venta_neta,
    ROUND(vm.costo, 2) as costo,
    ROUND(vm.venta_neta - vm.costo, 2) as margen,
    CASE 
        WHEN vm.venta_neta = 0 THEN 0
        ELSE ROUND(((vm.venta_neta - vm.costo) * 100.0 / vm.venta_neta), 2)
    END as margen_pct,
    
    -- Metas y cumplimiento
    ROUND(vm.meta_mensual_agencia, 2) as meta_mensual,
    CASE 
        WHEN vm.meta_mensual_agencia = 0 THEN NULL
        ELSE ROUND((vm.venta_neta * 100.0 / vm.meta_mensual_agencia), 2)
    END as cumplimiento_pct,
    
    -- Cálculo de comisión
    ROUND((vm.venta_neta * vend.comision_pct / 100), 2) as comision_transaccion,
    
    -- Información de estado
    vm.estado
    
FROM ventas_con_metas vm
LEFT JOIN vendedores vend ON vm.codigo_vendedor = vend.codigo_vendedor
LEFT JOIN distribuidoras_ref dr ON vm.codigo_distribuidora = dr.codigo_distribuidora
LEFT JOIN categorias_ref cr ON vend.categoria = cr.categoria
ORDER BY vm.fecha, vm.codigo_distribuidora, vm.codigo_agencia
"""

# Ejecutar consulta
sabana_df = con.execute(sabana_query).fetchdf()

print(f"✅ Sábana enriquecida creada: {len(sabana_df):,} registros")
print(f"📊 Columnas: {sabana_df.shape[1]}")

✅ Sábana enriquecida creada: 16,277 registros
📊 Columnas: 29


In [38]:
# EXPORTACIÓN FINAL DE LA SÁBANA ENRIQUECIDA

# Exportar a CSV
nombre_archivo = 'sabana_enriquecida.csv'
sabana_df.to_csv(nombre_archivo, index=False, encoding='utf-8')

display(sabana_df.head(3))

print(f"\n📋 COLUMNAS EXPORTADAS ({sabana_df.shape[1]}):")
for col in sabana_df.columns:
    print(f"  • {col}")


,id_transaccion,fecha,año,mes,codigo_distribuidora,nombre_distribuidora,region,codigo_agencia,codigo_vendedor,nombre_vendedor,...,canal_venta,tipo_cliente,venta_neta,costo,margen,margen_pct,meta_mensual,cumplimiento_pct,comision_transaccion,estado
0,TRX-1000005,2024-01-01,2024,1,DIST-001,Distribuidora Norte S.A.,Norte,AGN-101,V023,Tomás Aguilar,...,E-commerce,Persona Natural,2650447.56,1987138.06,663309.5,25.03,47323220.0,5.60,62550.56,Activo
1,TRX-1000018,2024-01-01,2024,1,DIST-001,Distribuidora Norte S.A.,Norte,AGN-101,V024,Paula Vega,...,Institucional,PYME,367957.57,265058.77,102898.8,27.96,47323220.0,0.78,10597.18,Activo
2,TRX-1000032,2024-01-01,2024,1,DIST-001,Distribuidora Norte S.A.,Norte,AGN-102,V027,Manuel Guerrero,...,E-commerce,Persona Natural,2650447.56,1987138.06,663309.5,25.03,35338034.0,7.50,54599.22,Activo



📋 COLUMNAS EXPORTADAS (29):
  • id_transaccion
  • fecha
  • año
  • mes
  • codigo_distribuidora
  • nombre_distribuidora
  • region
  • codigo_agencia
  • codigo_vendedor
  • nombre_vendedor
  • categoria_vendedor
  • fecha_ingreso
  • antiguedad_anios
  • comision_pct
  • vendedor_activo
  • estado_validacion_comision
  • codigo_producto
  • nombre_producto
  • categoria_producto
  • canal_venta
  • tipo_cliente
  • venta_neta
  • costo
  • margen
  • margen_pct
  • meta_mensual
  • cumplimiento_pct
  • comision_transaccion
  • estado


# ❓ PARTE 5: Preguntas Abiertas

**Objetivo:** Evaluar capacidad de análisis y comunicación.

In [34]:

print("""
1. ¿QUÉ PROBLEMAS DE CALIDAD DE DATOS ENCONTRÓ?

• Formatos inconsistentes: DIST_001 vs DIST-001, DIST-01 incompletos
• Valores nulos: 3-4% en códigos clave (distribuidora, agencia, vendedor)
• Inconsistencias entre tablas: 10 distribuidoras en metas sin ventas
• Valores atípicos extremos: Ventas promedio de $10.3M (escala inflada)
• Errores de escala: Datos aparentemente en "miles" o inflados para prueba

2. ¿QUÉ ESTRATEGIAS USÓ PARA RESOLVER LAS INCONSISTENCIAS EN LOS CRUCES?

• Normalización con CASE WHEN: DIST_001 → DIST-001, DIST-01 → DIST-001
• FULL OUTER JOIN: Para identificar registros sin correspondencia
• COALESCE(): Manejo de valores nulos en cruces
• Agrupamiento temporal: Antes del cruce para consistencia mensual
• Creación de vistas: metas_normalizadas para reutilización

3. SI TUVIERA QUE AUTOMATIZAR ESTE PROCESO MENSUALMENTE, ¿QUÉ CONSIDERACIONES TENDRÍA?

• Validaciones: Tests automáticos de calidad y consistencia
• Gobernanza: Versionado, auditoría y control de accesos
• Monitoreo: Dashboards con métricas de ejecución y calidad

4. ¿QUÉ VALIDACIONES AGREGARÍA ANTES DE PUBLICAR LOS DATOS PARA TABLEROS?

• Integridad: cálculos correctos
• Completitud: Sin nulos en campos obligatorios, cobertura completa
• Calidad: Rangos válidos,consistencia categórica
• Negocio: Detección de outliers, tendencias coherentes
• Checklist: Aprobaciones, backups, horarios no críticos
""")


1. ¿QUÉ PROBLEMAS DE CALIDAD DE DATOS ENCONTRÓ?

• Formatos inconsistentes: DIST_001 vs DIST-001, DIST-01 incompletos
• Valores nulos: 3-4% en códigos clave (distribuidora, agencia, vendedor)
• Inconsistencias entre tablas: 10 distribuidoras en metas sin ventas
• Valores atípicos extremos: Ventas promedio de $10.3M (escala inflada)
• Errores de escala: Datos aparentemente en "miles" o inflados para prueba

2. ¿QUÉ ESTRATEGIAS USÓ PARA RESOLVER LAS INCONSISTENCIAS EN LOS CRUCES?

• Normalización con CASE WHEN: DIST_001 → DIST-001, DIST-01 → DIST-001
• FULL OUTER JOIN: Para identificar registros sin correspondencia
• COALESCE(): Manejo de valores nulos en cruces
• Agrupamiento temporal: Antes del cruce para consistencia mensual
• Creación de vistas: metas_normalizadas para reutilización

3. SI TUVIERA QUE AUTOMATIZAR ESTE PROCESO MENSUALMENTE, ¿QUÉ CONSIDERACIONES TENDRÍA?

• Validaciones: Tests automáticos de calidad y consistencia
• Gobernanza: Versionado, auditoría y control de acces